In [155]:
print('Ritu')

Ritu


In [156]:
from langchain_ai21.chat_models import ChatAI21
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import TypedDict,Annotated, List, Dict
from dotenv import load_dotenv
import json

load_dotenv()

model = ChatAI21(model = 'jamba-mini-1.7-2025-07')
searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)


In [157]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 

In [158]:
DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def except_detail_prompt(detail_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{detail_text}

Please convert this into valid JSON with the following structure:
{{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {{
            "key_point":
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt

In [188]:
def is_valid_brackets(s: str) -> bool:
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack:
         s += bracket_map[char]
     return s

In [189]:
def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text


In [190]:
class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    num_slides: int

In [191]:
def chat_node(state: PptState):
    """Main chat node that processess messages"""
    messages = state["messages"]
    result = model_with_tools.invoke(messages)
    return {'messages':result}

In [192]:
def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state["topic"]
    num_slides = state['num_slides']

    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )

    messages = [OUTLINE_SYSTEM_PORMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('generate_outline_node result',type(result.content))
    try:
        outline = json_to_python(result.content)
        outline = json.loads(outline)
    except json.JSONDecodeError:
        
        outline = json.loads(is_valid_brackets(outline))
        # fix_prompt = except_outline_prompt(result.content)
        # try:
        #     fix_result = model.invoke([fix_prompt])
        #     fixed_text = fix_result.content.strip()
        #     outline = json_to_python(fixed_text)
        #     outline = json.loads(outline)
        # except json.JSONDecodeError:
        #     outline = {
        #         'title': topic,
        #         'total_slide': num_slides,
        #         'slides': []
        #     }
    print('outline',outline['slides'])

    return {
        'messages':[result],
        'outline':outline,
        'current_slide_index':0
            }

In [193]:
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    if current_index > state['num_slides']:
        return
    
    # print('outline',type(outline),outline)
    print('current_index',current_index)
    current_slide = outline['slides'][int(current_index)]
    # print('current_slide',current_slide)
    # print('slide_title',current_slide['slide_title'])
    # print('key_points',current_slide['key_points'])
    # print('content_type',current_slide['content_type'])

    prompt = HumanMessage(
        content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        print('before is_valid_brackets',detailed_slide)
        print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except:
            fix_prompt = except_detail_prompt(detailed_slide)
            try:
                fix_result = model.invoke([fix_prompt])
                fixed_text = fix_result.content.strip()
                detailed_slide = json_to_python(fixed_text)
                detailed_slide = json.loads(is_valid_brackets(detailed_slide))
            except json.JSONDecodeError as e:
                print('last erroe',e)
                print('detailed_slide',detailed_slide)
                detailed_slide = {
                    'slide_number': current_slide['slide_number'],
                    'slide_title': current_slide['slide_title'],
                    'detailed_content':[],
                    'row_response': detailed_slide
                }
    
    detailed_slides = state.get('detailed_slides',[])
    detailed_slides.append(detailed_slide)
    return {
        'messages':[result],
        'detailed_slides':detailed_slides,
        'current_slide_index': current_index +1
    }

In [194]:
def should_coutine_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index',0)
    total_slides = len(state.get('outline',{}).get('slides',[]))

    if current_index < total_slides:
        return "continue"
    else:
        return "end"

In [195]:
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node("generate_slide_detail",generate_slide_detail_node)
workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
workflow.add_edge("generate_outline","generate_slide_detail")
workflow.add_conditional_edges(
    "generate_slide_detail",should_coutine_slides,
    {
        "continue":"generate_slide_detail",
        "end":END
    }

)


ppt_generator = workflow.compile()

In [196]:
initial_state = {
        'messages': [],
        'outline': {},
        'detailed_slides': [],
        'current_slide_index': 0,
        'topic': 'What is photosynthesis in plants',
        'num_slides': 5
    }
result = ppt_generator.invoke(initial_state)

outline [{'slide_number': 1, 'slide_title': 'Introduction to Photosynthesis', 'key_points': ['Definition of Photosynthesis', 'Importance of Photosynthesis', 'Basic Concept of Photosynthesis'], 'content_type': 'introduction'}, {'slide_number': 2, 'slide_title': 'Process of Photosynthesis', 'key_points': ['Light Absorption', 'Carbon Dioxide Uptake', 'Water Splitting', 'Formation of Glucose'], 'content_type': 'explanation'}, {'slide_number': 3, 'slide_title': 'Key Components of Photosynthesis', 'key_points': ['Chlorophyll', 'Chloroplasts', 'Water and Sunlight'], 'content_type': 'comparison'}, {'slide_number': 4, 'slide_title': 'Significance of Photosynthesis', 'key_points': ['Production of Oxygen', 'Food Production', 'Energy Source for Plants'], 'content_type': 'explanation'}, {'slide_number': 5, 'slide_title': 'Conclusion and Applications', 'key_points': ['Summary of Photosynthesis', 'Applications in Technology and Agriculture', 'Future Perspectives'], 'content_type': 'conclusion'}]
curr

In [200]:
result['detailed_slides']

[{'slide_number': 1,
  'slide_title': 'Introduction to Photosynthesis',
  'detailed_content': [{'key_point': {'title': 'Definition of Photosynthesis',
     'content': 'Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy from the sun into chemical energy stored in glucose. This process is essential for life on Earth as it provides the primary source of energy for almost all living organisms.',
     'example': 'For example, in plants, photosynthesis occurs in the chloroplasts, which contain the pigment chlorophyll, capturing sunlight to drive the process.'}},
   {'key_point': {'title': 'Importance of Photosynthesis',
     'content': 'Photosynthesis is crucial for maintaining the balance of oxygen and carbon dioxide in the atmosphere. It produces oxygen as a byproduct, which is vital for the survival of aerobic organisms. Additionally, it forms the basis of the food chain, as plants are the primary producers, providing energy for herbivores, 

In [170]:
s = """{
    "slide_number": "The Process of Photosynthesis",
    "slide_title": "The Process of Photosynthesis",
    "detailed_content": [
        {
            "key_point": "Light Absorption by Chlorophyll",
            "explanation": "Chlorophyll, the green pigment in plants, absorbs light primarily from the blue and red wavelengths of sunlight. This light energy is then used to excite electrons, initiating the photosynthetic process. For example, in the chloroplasts, chlorophyll molecules in the thylakoid membranes capture photons and transfer energy to electrons. This process is crucial for converting light into chemical energy.",
            "examples": "An example of this is seen in the absorption spectrum of chlorophyll, which peaks in the blue (around 430 nm) and red (around 680 nm) regions of the visible light spectrum."
        },
        {
            "key_point": "Role of Water and Carbon Dioxide",
            "explanation": "Water (H₂O) provides electrons and protons necessary for the light-dependent reactions, while carbon dioxide (CO₂) is essential for the Calvin cycle, where glucose is synthesized. Water molecules are split during photosynthesis, releasing oxygen as a byproduct. This process is vital for the production of energy-rich molecules. For instance, the splitting of water molecules in the thylakoid membranes provides the electrons needed for the electron transport chain.",
            "examples": "An example of this is the Calvin cycle, where CO₂ is fixed into glucose through a series of enzyme-driven reactions."
        },
        {
            "key_point": "Formation of Glucose",
            "explanation": "The glucose (C₆H₁₂O₆) produced during photosynthesis serves as an energy source for the plant and can be stored for later use. This process involves the Calvin cycle, where CO₂ is fixed into glucose using the energy from ATP and NADPH produced in the light-dependent reactions. For example, in the stroma of the chloroplast, the enzyme Rubisco catalyzes the fixation of CO₂ into a 3-carbon compound, which is then converted into glucose.",
            "examples": "An example of this is seen in the storage of glucose as starch in plant"}]}"""

In [171]:
# s = is_valid_brackets(s)
# print(s)
json.loads(s)

{'slide_number': 'The Process of Photosynthesis',
 'slide_title': 'The Process of Photosynthesis',
 'detailed_content': [{'key_point': 'Light Absorption by Chlorophyll',
   'explanation': 'Chlorophyll, the green pigment in plants, absorbs light primarily from the blue and red wavelengths of sunlight. This light energy is then used to excite electrons, initiating the photosynthetic process. For example, in the chloroplasts, chlorophyll molecules in the thylakoid membranes capture photons and transfer energy to electrons. This process is crucial for converting light into chemical energy.',
   'examples': 'An example of this is seen in the absorption spectrum of chlorophyll, which peaks in the blue (around 430 nm) and red (around 680 nm) regions of the visible light spectrum.'},
  {'key_point': 'Role of Water and Carbon Dioxide',
   'explanation': 'Water (H₂O) provides electrons and protons necessary for the light-dependent reactions, while carbon dioxide (CO₂) is essential for the Calvin

In [181]:
s = """{ slide_number: 1
slide_title: Introduction to Photosynthesis
detailed_content:
  - key_point:
        explanation: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy stored in glucose.
        examples: "For example, in plants, photosynthesis occurs in the chloroplasts, which contain the pigment chlorophyll. This process is essential for the production of food and oxygen."
        statistics: "Approximately 10% of the Earth's surface is covered by oceans, where photosynthetic organisms like phytoplankton play a crucial role in regulating the planet's climate."
  - key_point:
        explanation: Photosynthesis is crucial for plants as it provides the energy needed for growth and reproduction. For the ecosystem, it is vital as it forms the base of the food chain and helps regulate the carbon cycle.
        examples: "Without photosynthesis, plants would not be able to produce the glucose necessary for their survival, leading to a collapse of the food chain."
        statistics: "Plants produce about 115-120 billion tons of biomass per year, with about 60% of this biomass coming from photosynthesis."
  - key_point:
        explanation: The photosynthesis process involves three main stages: light absorption, the light-dependent reactions, and the Calvin cycle.
        examples: "In the light-dependent reactions, light energy is used to split water molecules, releasing oxygen and generating ATP and NADPH. These products are then used in the Calvin cycle to produce glucose."
        statistics: "The efficiency of photosynthesis varies by plant type, with C4 plants being more efficient in hot and dry environments, while C3 plants are more efficient in cooler environments."}"""

In [ ]:
s = except_detail_prompt(s)
print(s)
s = model.invoke([s])


content='The following text should be valid JSON but has formatting issues:\n\n{ slide_number: 1\nslide_title: Introduction to Photosynthesis\ndetailed_content:\n  - key_point:\n        explanation: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy stored in glucose.\n        examples: "For example, in plants, photosynthesis occurs in the chloroplasts, which contain the pigment chlorophyll. This process is essential for the production of food and oxygen."\n        statistics: "Approximately 10% of the Earth\'s surface is covered by oceans, where photosynthetic organisms like phytoplankton play a crucial role in regulating the planet\'s climate."\n  - key_point:\n        explanation: Photosynthesis is crucial for plants as it provides the energy needed for growth and reproduction. For the ecosystem, it is vital as it forms the base of the food chain and helps regulate the carbon cycle.\n        examples: "Without photosynth

In [183]:
print(s.content)

{
    "slide_number": 1,
    "slide_title": "Introduction to Photosynthesis",
    "detailed_content": [
        {
            "key_point": {
                "explanation": "Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy stored in glucose.",
                "examples": "For example, in plants, photosynthesis occurs in the chloroplasts, which contain the pigment chlorophyll. This process is essential for the production of food and oxygen.",
                "statistics": "Approximately 10% of the Earth's surface is covered by oceans, where photosynthetic organisms like phytoplankton play a crucial role in regulating the planet's climate."
            }
        },
        {
            "key_point": {
                "explanation": "Photosynthesis is crucial for plants as it provides the energy needed for growth and reproduction. For the ecosystem, it is vital as it forms the base of the food chain and helps regulate the car